In [10]:
import json

import requests
from dotenv import load_dotenv
import openai

load_dotenv()

client = openai.OpenAI()

MOVIE_API_BASE_URL = "https://nomad-movies-2.nomadcoders.workers.dev"

In [11]:
# 실제 API를 호출하는 도구(tool) 함수들


def simplify_movie(movie):
    return {
        "id": movie.get("id"),
        "title": movie.get("title"),
        "overview": movie.get("overview"),
        "release_date": movie.get("release_date"),
        "vote_average": movie.get("vote_average"),
    }


def get_popular_movies():
    response = requests.get(f"{MOVIE_API_BASE_URL}/movies")
    response.raise_for_status()
    movies = response.json()
    return [simplify_movie(movie) for movie in movies]


def get_movie_details(id):
    response = requests.get(f"{MOVIE_API_BASE_URL}/movies/{id}")
    response.raise_for_status()
    movie = response.json()
    return {
        "id": movie.get("id"),
        "title": movie.get("title"),
        "original_title": movie.get("original_title"),
        "overview": movie.get("overview"),
        "release_date": movie.get("release_date"),
        "runtime": movie.get("runtime"),
        "genres": [genre["name"] for genre in movie.get("genres", [])],
        "vote_average": movie.get("vote_average"),
        "tagline": movie.get("tagline"),
        "status": movie.get("status"),
    }


def get_similar_movies(id):
    response = requests.get(f"{MOVIE_API_BASE_URL}/movies/{id}/similar")
    response.raise_for_status()
    movies = response.json()
    return [simplify_movie(movie) for movie in movies]

In [12]:
# OpenAI tools 파라미터에 전달할 함수(도구) 스펙

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_popular_movies",
            "description": "현재 인기 있는 영화 목록을 가져옵니다.",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": [],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details",
            "description": "영화 ID로 특정 영화의 상세 정보(줄거리, 장르, 평점, 런타임 등)를 가져옵니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "integer",
                        "description": "영화의 TMDB ID",
                    },
                },
                "required": ["id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_similar_movies",
            "description": "영화 ID를 기반으로 비슷한 영화 목록을 가져옵니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "integer",
                        "description": "영화의 TMDB ID",
                    },
                },
                "required": ["id"],
            },
        },
    },
]

# 도구 이름 -> 실제 함수 매핑
available_functions = {
    "get_popular_movies": get_popular_movies,
    "get_movie_details": get_movie_details,
    "get_similar_movies": get_similar_movies,
}

In [13]:
messages = [
    {
        "role": "system",
        "content": (
            "당신은 영화 전문 추천 챗봇입니다. "
            "사용자의 질문에 답하기 위해 필요할 때마다 제공된 도구(tool)를 호출해 "
            "실제 영화 데이터를 가져온 뒤, 그 결과를 바탕으로 한국어로 자연스럽게 답변하세요. "
            "이전 대화에서 언급된 영화나 ID를 기억하고 맥락에 맞게 활용하세요."
        ),
    }
]

In [14]:
def call_ai():
    while True:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=tools,
        )
        message = response.choices[0].message
        messages.append(message.model_dump(exclude_none=True))

        if not message.tool_calls:
            print(f"AI: {message.content}")
            return message.content

        for tool_call in message.tool_calls:
            function_name = tool_call.function.name
            function_args = json.loads(tool_call.function.arguments)
            print(f"[도구 호출] {function_name}({function_args})")

            function_to_call = available_functions[function_name]
            function_response = function_to_call(**function_args)

            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": json.dumps(function_response, ensure_ascii=False),
                }
            )

In [ ]:
while True:
    user_input = input("메시지를 입력하세요 (종료: q): ")
    if user_input.lower() == "q":
        break
    print(f"User: {user_input}")
    messages.append({"role": "user", "content": user_input})
    call_ai()

User: 지금 인기 있는 영화 알려줘
[도구 호출] get_popular_movies({})
AI: 현재 인기 있는 영화 목록은 다음과 같습니다:

1. **Peddi**
   - 줄거리: 1980년대 안드라프라데시의 시골에서 한 열정적인 마을 사람이 스포츠를 통해 자신의 공동체를 단합시켜 강력한 적과의 자존심을 지킵니다.
   - 평점: 6.339
   - 개봉일: 2026-06-03

2. **Obsession**
   - 줄거리: 신비로운 "원 위시 윌로우"를 부수고 사랑하는 사람의 마음을 얻으려는 한 로맨티스트가 원하는 것을 얻게 되지만, 그 욕망이 어둡고 음습한 대가를 가져온다는 것을 발견하게 됩니다.
   - 평점: 7.905
   - 개봉일: 2026-05-13

3. **Hai Jawani Toh Ishq Hona Hai**
   - 줄거리: 결혼을 포기한 Jass가 새로운 해외 로맨스에 휘말리게 되면서 충격적인 사실로 인해 사랑과 충성심, 헌신의 진정한 의미에 마주하게 됩니다.
   - 평점: 5.4
   - 개봉일: 2026-06-04

4. **The Unknown Man**
   - 줄거리: 플랑드르 작가 루이스가 영감을 얻기 위해 코트 다쥐르에서 자기를 격리하려 합니다.
   - 평점: 8.063
   - 개봉일: 2021-10-16

5. **Mortal Kombat II**
   - 줄거리: 팬들이 좋아하는 챔피언들이 이제 Johnny Cage와 함께 Earth's defenders의 존재를 위협하는 Shao Kahn의 어두운 통치에 맞서 싸우는 궁극적이고 잔인한 전투에 나섭니다.
   - 평점: 7.958
   - 개봉일: 2026-05-06

더 많은 영화 정보를 원하시면 말씀해 주세요!
User: Peddi에 대해 더 알려줘
[도구 호출] get_movie_details({'id': 1057265})
AI: **Peddi** (원제: పెద్ది)

- **줄거리**: 1980년대 안드라프라데시의 시골에서 한 열정적인 마을 사